In [3]:
import requests
from requests_oauthlib import OAuth1
import os

# Read credentials from ENV
ACCOUNT_ID = os.getenv("ACCOUNT_ID")

CONSUMER_KEY = os.getenv("CONSUMER_KEY")
CONSUMER_SECRET = os.getenv("CONSUMER_SECRET")
TOKEN_ID = os.getenv("TOKEN_ID")
TOKEN_SECRET = os.getenv("TOKEN_SECRET")

auth = OAuth1(
    CONSUMER_KEY,
    CONSUMER_SECRET,
    TOKEN_ID,
    TOKEN_SECRET,
    signature_method="HMAC-SHA256",
    realm=ACCOUNT_ID
)

url = f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/record/v1/metadata-catalog"

response = requests.get(
    url,
    auth=auth,
    headers={"Content-Type": "application/json"}
)

print(response.status_code)
print(response.text)

401
{"type":"https://www.rfc-editor.org/rfc/rfc9110.html#section-15.5.2","title":"Unauthorized","status":401,"o:errorDetails":[{"detail":"Invalid login attempt. For more details, see the Login Audit Trail in the NetSuite UI at Setup > Users/Roles > User Management > View Login Audit Trail.","o:errorCode":"INVALID_LOGIN"}]}



In [4]:
# Check all your active tokens
r = requests.get(
    f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/record/v1/metadata-catalog",
    auth=auth,
    headers={"Content-Type": "application/json"}
)
print("Metadata:", r.status_code)

# If metadata works, the token is valid
# The role ID in use will be in this response
if r.status_code == 200:
    print("✅ Token is valid and working")
    print("The role permission issue is confirmed")
else:
    print("❌ Token itself is invalid - regenerate credentials")

Metadata: 401
❌ Token itself is invalid - regenerate credentials


In [5]:
import requests
from requests_oauthlib import OAuth1
import os



auth = OAuth1(
    CONSUMER_KEY,
    CONSUMER_SECRET,
    TOKEN_ID,
    TOKEN_SECRET,
    signature_method="HMAC-SHA256",
    realm=ACCOUNT_ID
)

BASE_URL = f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/record/v1"

# Step 1: confirm metadata still works
r = requests.get(
    f"{BASE_URL}/metadata-catalog",
    auth=auth,
    headers={"Content-Type": "application/json"}
)
print("Metadata status:", r.status_code)

# Step 2: try currency
r2 = requests.get(
    f"{BASE_URL}/currency",
    auth=auth,
    headers={"Content-Type": "application/json"},
    params={"limit": 1}
)
print("Currency status:", r2.status_code)
print("Currency response:", r2.text[:300])

Metadata status: 401
Currency status: 401
Currency response: {"type":"https://www.rfc-editor.org/rfc/rfc9110.html#section-15.5.2","title":"Unauthorized","status":401,"o:errorDetails":[{"detail":"Invalid login attempt. For more details, see the Login Audit Trail in the NetSuite UI at Setup > Users/Roles > User Management > View Login Audit Trail.","o:errorCode


In [ ]:
ACCOUNT_ID      = "1227893"
auth = OAuth1(
    CONSUMER_KEY,
    CONSUMER_SECRET,
    TOKEN_ID,
    TOKEN_SECRET,
    signature_method="HMAC-SHA256",
    realm=ACCOUNT_ID
)

# Test immediately
r = requests.get(
    f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/record/v1/currency",
    auth=auth,
    headers={"Content-Type": "application/json"},
    params={"limit": 1}
)
print(r.status_code)
print(r.text[:300])

200
{"links":[{"rel":"next","href":"https://1227893.suitetalk.api.netsuite.com/services/rest/record/v1/currency?limit=1&offset=1"},{"rel":"last","href":"https://1227893.suitetalk.api.netsuite.com/services/rest/record/v1/currency?limit=1&offset=4"},{"rel":"self","href":"https://1227893.suitetalk.api.nets


In [20]:
import requests
from requests_oauthlib import OAuth1
import pandas as pd

def suiteql(query):
    r = requests.post(
        f"https://{ACCOUNT_ID}.suitetalk.api.netsuite.com/services/rest/query/v1/suiteql",
        auth=auth,
        headers={"Content-Type": "application/json", "Prefer": "transient"},
        json={"q": query}
    )
    if r.status_code == 200:
        items = r.json().get("items", [])
        return pd.DataFrame(items)
    else:
        print("Error:", r.text)
        return None

# ── Currency ──
df_currency = suiteql("""
    SELECT 
        id,
        name,
        symbol,
        exchangerate,
        isbasecurrency,
        isinactive
    FROM currency
    ORDER BY name
""")

print("=" * 60)
print("CURRENCY TABLE")
print("=" * 60)
print(df_currency.to_string(index=False))
print(f"\nTotal rows: {len(df_currency)}")

CURRENCY TABLE
links      exchangerate id isbasecurrency isinactive name symbol
   [] 0.547822133109822  3              F          F  CAD    CAD
   []  0.95872083631136  5              F          F  CHF    CHF
   []           0.86908  4              F          F  EUR    EUR
   []                 1  1              T          F  GBP    GBP
   [] 0.748329354715597  2              F          F  USD    USD

Total rows: 5


In [28]:
df_test = suiteql("""
SELECT *
FROM transaction
WHERE ROWNUM <= 1
""")

print(df_test)

  links abbrevtype balsegstatus billingaddress billingstatus   closedate  \
0    []        INV            5            116             F  30/05/2025   

  createdby createddate createdfrommerge currency  ...    trandate  \
0         3  10/05/2025                F        1  ...  01/06/2024   

   trandisplayname  tranid transactionnumber      type  \
0  Invoice #114493  114493        CUSTINVC22  CustInvc   

  typebaseddocumentnumber userevenuearrangement visibletocustomer void voided  
0                  114493                     T                 T    F      F  

[1 rows x 88 columns]


In [29]:
df_invoices = suiteql("""
SELECT 
    id,
    tranid,
    entity,
    trandate,
    status,
    type
FROM transaction
WHERE type = 'CustInvc'
AND ROWNUM <= 20
ORDER BY trandate DESC
""")

print(df_invoices)

   links entity    id status    trandate  tranid      type
0     []    754  9304      B  28/08/2025  115692  CustInvc
1     []    172  9322      B  19/08/2025  115691  CustInvc
2     []    740  9295      B  01/08/2025  115664  CustInvc
3     []    141  9309      B  01/08/2025  115645  CustInvc
4     []    208  9300      B  31/07/2025  115627  CustInvc
5     []    119  9308      B  31/07/2025  115630  CustInvc
6     []    211  9303      B  30/06/2025  115569  CustInvc
7     []    135  9313      B  01/06/2025  115503  CustInvc
8     []    158  9302      B  01/06/2025  115518  CustInvc
9     []    159  9299      B  31/05/2025  115460  CustInvc
10    []    161  9311      B  31/05/2025  115461  CustInvc
11    []    183  9305      B  01/05/2025  115413  CustInvc
12    []    758  9315      B  01/05/2025  115410  CustInvc
13    []    141  9323      B  30/04/2025  115370  CustInvc
14    []    178  9320      B  31/03/2025  115326  CustInvc
15    []    735  9317      B  01/03/2025  115217  CustIn

In [30]:
df_customers = suiteql("""
SELECT 
    id,
    entityid,
    email,
    phone
FROM customer
WHERE ROWNUM <= 20
""")

print(df_customers)

   links                                              email entityid   id  \
0     []                      louise@logisticplanning.co.uk       65  177   
1     []                        terry@maltbylogistics.co.uk       67  179   
2     []                     invoices@maritimetransport.com       69  181   
3     []                      vishal@mgmhaulageservices.com       71  183   
4     []                        abigoulding@nwtransport.com       73  185   
5     []                         bharat@mbssecurities.co.uk       75  187   
6     []                      amanda@gippingtransport.co.uk       77  189   
7     []                        brendan.swift@pdports.co.uk       79  191   
8     []                 purchaseledger@porticoshipping.com       80  192   
9     []                                                NaN       82  194   
10    []                   accounts@proactive-logistics.com       83  195   
11    []                accounts@redskytransportation.co.uk       85  197   